# Песочница паттернов GoF: Strategy, Factory Method, Observer

**Выполнила:** Калинина И.А., группа ГИ-41, вариант 9 (пункт таможенного оформления).

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java. Каждый пример представлен **одним файлом** `.java` и запускается без отдельной компиляции командой `java Файл.java` (запуск исходного файла, Java 11 и выше).

**Идея задания.** Паттерн изучается не по описанию, а по разнице: сначала запускается код «до» — работающий, но жёстко связанный; затем тот же сценарий переписывается на паттерн, и проверки доказывают, что **поведение не изменилось**, а **новое требование добавляется без правки старого кода**.

**Состав работы**

| Раздел | Паттерн | Что делает студент |
|---|---|---|
| 2 | Strategy | **образец** — разобран полностью, нужно только запустить и разобраться |
| 3 | Factory Method | задача: самостоятельно переписать код «до» |
| 4 | Observer | задача: самостоятельно переписать код «до» |
| 5–6 | — | сравнение, выводы, контрольные вопросы |

**Правила оформления, от которых зависит автопроверка**

1. Имена файлов строго: `TariffAfter.java`, `FactoryAfter.java`, `ObserverAfter.java`.
2. В каждом файле **первым** идёт открытый класс с методом `main` — иначе запуск исходного файла не работает.
3. Каждый файл сам себя проверяет: печатает строки `[OK]` / `[FAIL]` и бросает `AssertionError`, если хотя бы одна проверка не пройдена.
4. Идентификаторы — латиницей и описательно (`tariffStrategy`, а не `ts`), комментарии — по-русски.
5. К каждому паттерну — диаграмма классов в файле `*.puml`.

**Сдача.** Все файлы `.java`, `.puml`, сам блокнот и вывод ячейки самопроверки — в pull request.


## 1. Подготовка среды

In [ ]:
# 1.1 Проверить, какая Java доступна в среде.
# Нужна версия 11 или выше: только с неё работает запуск одного исходного файла (java Файл.java).
!java -version
!javac -version

Если версия ниже 11, запуск одного исходного файла недоступен — всюду в блокноте вместо `!java Файл.java` используйте два шага: `!javac Файл.java && java Файл`.

Проверка «живого» запуска (ячейка 1.2 записывает файл, 1.3 — запускает):

In [ ]:
%%writefile Hello.java
public class Hello {
    public static void main(String[] args) {
        System.out.println("Среда готова, Java работает: " + System.getProperty("java.version"));
    }
}

In [ ]:
!java Hello.java

---

## 2. Образец: Strategy (разобран полностью)

**Сценарий.** Расчёт платы за перевозку груза. Плата зависит от вида отправки: повагонная, контейнерная, маршрутная. Формулы учебные.

### 2.1 Код «до»: один метод знает про все виды отправок

In [ ]:
%%writefile TariffBefore.java
// Код «до»: вид отправки разбирается условным оператором внутри одного метода.
public class TariffBefore {

    public static void main(String[] args) {
        System.out.printf("повагонная   : %10.2f%n", cost("WAGON", 60, 500));
        System.out.printf("контейнерная : %10.2f%n", cost("CONTAINER", 24, 500));
        System.out.printf("маршрутная   : %10.2f%n", cost("BLOCK_TRAIN", 3000, 500));
        System.out.printf("рефрижератор : %10.2f%n", cost("REFRIGERATED", 60, 500));
    }

    static double cost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;   // про новый вид отправки метод не знает
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

In [ ]:
!java TariffBefore.java

### 2.2 Разбор: что именно плохо

Запустив код «до», обратите внимание:

* рефрижераторная отправка дала **0.00** — метод о ней не знает, и единственный способ это исправить — залезть внутрь уже работающего метода;
* каждый новый вид отправки удлиняет один и тот же метод, то есть нарушает **принцип открытости/закрытости** (открыт для расширения, закрыт для изменения);
* метод одновременно отвечает за выбор варианта и за сам расчёт — нарушение **единственной ответственности**;
* протестировать один вид отправки отдельно от остальных нельзя.

**Что делает Strategy.** Каждый вариант расчёта становится отдельным классом с общим интерфейсом, а вызывающий код работает с интерфейсом и больше не знает перечня вариантов.

### 2.3 Код «после»: вариант расчёта вынесен в стратегию

In [ ]:
%%writefile TariffAfter.java
// Код «после»: каждый вид отправки — отдельная стратегия расчёта.
import java.util.LinkedHashMap;
import java.util.Map;

public class TariffAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {

        // Каталог стратегий: вызывающий код собирает его, но не содержит формул
        Map<String, TariffStrategy> tariffCatalog = new LinkedHashMap<>();
        tariffCatalog.put("WAGON", new WagonTariff());
        tariffCatalog.put("CONTAINER", new ContainerTariff());
        tariffCatalog.put("BLOCK_TRAIN", new BlockTrainTariff());

        TariffCalculator tariffCalculator = new TariffCalculator();

        System.out.println("--- Расчёт по каталогу стратегий (масса 60 т, расстояние 500 км) ---");
        for (Map.Entry<String, TariffStrategy> catalogEntry : tariffCatalog.entrySet()) {
            double calculatedCost = tariffCalculator.calculate(catalogEntry.getValue(), 60, 500);
            System.out.printf("%-12s %-28s %10.2f%n",
                    catalogEntry.getKey(), catalogEntry.getValue().title(), calculatedCost);
        }

        System.out.println();
        System.out.println("--- Характеризующие тесты: поведение совпадает с кодом «до» ---");
        check("повагонная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("WAGON"), 60, 500),
                           legacyCost("WAGON", 60, 500)));
        check("контейнерная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("CONTAINER"), 24, 500),
                           legacyCost("CONTAINER", 24, 500)));
        check("маршрутная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("BLOCK_TRAIN"), 3000, 500),
                           legacyCost("BLOCK_TRAIN", 3000, 500)));

        System.out.println();
        System.out.println("--- Тест открытости/закрытости: новый вид отправки без правки существующих классов ---");
        TariffStrategy refrigeratedTariff = new TariffStrategy() {
            public double cost(double massTons, int distanceKm) {
                return 1.80 * massTons * distanceKm / 100.0 + 600.0;
            }
            public String title() {
                return "рефрижераторная отправка";
            }
        };
        double refrigeratedCost = tariffCalculator.calculate(refrigeratedTariff, 60, 500);
        System.out.printf("%-12s %-28s %10.2f%n", "REFRIGERATED", refrigeratedTariff.title(), refrigeratedCost);
        check("новая стратегия рассчитана, класс TariffCalculator не изменён", refrigeratedCost > 0.0);
        check("в коде «до» тот же вид отправки давал нуль", legacyCost("REFRIGERATED", 60, 500) == 0.0);

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    // --- служебное ---

    static boolean equalMoney(double firstValue, double secondValue) {
        return Math.abs(firstValue - secondValue) < 0.01;
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    // Старая реализация сохранена только как эталон поведения для характеризующих тестов
    static double legacyCost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

// --- участники паттерна Strategy ---

/** Стратегия: общий интерфейс всех вариантов расчёта. */
interface TariffStrategy {
    double cost(double massTons, int distanceKm);
    String title();
}

/** Контекст: умеет считать, но не знает, какие бывают виды отправок. */
class TariffCalculator {
    double calculate(TariffStrategy tariffStrategy, double massTons, int distanceKm) {
        double calculatedValue = tariffStrategy.cost(massTons, distanceKm);
        return Math.round(calculatedValue * 100.0) / 100.0;
    }
}

class WagonTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 1.20 * massTons * distanceKm / 100.0 + 150.0;
    }
    public String title() {
        return "повагонная отправка";
    }
}

class ContainerTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.90 * massTons * distanceKm / 100.0 + 400.0;
    }
    public String title() {
        return "контейнерная отправка";
    }
}

class BlockTrainTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.70 * massTons * distanceKm / 100.0;
    }
    public String title() {
        return "маршрутная отправка";
    }
}

In [ ]:
!java TariffAfter.java

### 2.4 Диаграмма классов образца

Установка PlantUML занимает около минуты и выполняется один раз за сеанс.

In [ ]:
!apt-get -qq install -y plantuml > /dev/null && echo "PlantUML установлен"

In [ ]:
%%writefile strategy.puml
@startuml
title Strategy: расчёт платы за перевозку

interface TariffStrategy {
  +cost(massTons, distanceKm) : double
  +title() : String
}

class TariffCalculator {
  +calculate(tariffStrategy, massTons, distanceKm) : double
}

class WagonTariff
class ContainerTariff
class BlockTrainTariff

TariffCalculator o--> TariffStrategy : использует
TariffStrategy <|.. WagonTariff
TariffStrategy <|.. ContainerTariff
TariffStrategy <|.. BlockTrainTariff
@enduml

In [ ]:
!plantuml -tpng strategy.puml
from IPython.display import Image
Image("strategy.png")

### 2.5 Вопросы к образцу (ответить текстом в этой ячейке)

1. Какие классы пришлось бы изменить в коде «до» и в коде «после», чтобы добавить рефрижераторную отправку?
   **Ответ:** В коде «до» менять пришлось бы метод `cost` в `TariffBefore`: добавить ещё одну ветку `if`. Других классов там нет, вся формула сидит в одном месте. В коде «после» `TariffCalculator`, `WagonTariff`, `ContainerTariff` и `BlockTrainTariff` я не трогаю. Нужен новый класс с интерфейсом `TariffStrategy` и строка, которая кладёт его в каталог, как рефрижератор уже сделан в `main`.
2. Зачем в файле оставлен метод `legacyCost`, если он больше не используется в расчёте?
   **Ответ:** Это старый расчёт, оставленный как эталон. Тесты сравнивают с ним повагонную, контейнерную и маршрутную отправку, чтобы после переписывания суммы не разъехались. В самом расчёте метод больше не участвует. Для рефрижератора он специально даёт 0, и тест как раз проверяет, что код «до» этот вид не умел.
3. Что произойдёт, если в каталоге не окажется запрошенного вида отправки, и как это следует обработать?
   **Ответ:** `tariffCatalog.get` вернёт `null`, и вызов `calculate` упадёт с `NullPointerException`: стратегии нет, считать не с чего. Это честнее, чем в коде «до», где незнакомый тип тихо давал 0.00. Перед расчётом нужно проверить, что вид отправки есть в каталоге, и сообщить об ошибке, а не подставлять нулевую плату.


---

## 3. Задача 1: Factory Method

**Сценарий.** Описание вагона по его типу: крытый, платформа, цистерна. Ниже — код «до», построенный на той же ошибке: условный оператор вместо полиморфизма.

### 3.1 Код «до» (запустить, не изменяя)

In [ ]:
%%writefile FactoryBefore.java
// Код «до»: создание и описание объекта подменены разбором строкового типа.
public class FactoryBefore {

    public static void main(String[] args) {
        System.out.println(describe("COVERED", 68));
        System.out.println(describe("PLATFORM", 71));
        System.out.println(describe("TANK", 66));
        System.out.println(describe("HOPPER", 70));   // новый тип — обработчика нет
    }

    static String describe(String wagonType, double capacityTons) {
        if (wagonType.equals("COVERED")) {
            return "Крытый вагон: грузоподъёмность " + capacityTons + " т, тарные и штучные грузы";
        } else if (wagonType.equals("PLATFORM")) {
            return "Платформа: грузоподъёмность " + capacityTons + " т, длинномерные и тяжеловесные грузы";
        } else if (wagonType.equals("TANK")) {
            return "Цистерна: грузоподъёмность " + capacityTons + " т, наливные грузы";
        }
        return "Неизвестный тип вагона: " + wagonType;
    }
}

In [ ]:
!java FactoryBefore.java

### 3.2 Задание

Переписать сценарий на **Factory Method** в файле `FactoryAfter.java`. Требования:

1. Общий тип продукта: интерфейс или абстрактный класс `Wagon` с методами `description()` и `capacityTons()`.
2. Не менее трёх конкретных продуктов: `CoveredWagon`, `Platform`, `TankWagon`.
3. Создатель с фабричным методом: абстрактный класс или интерфейс `WagonCreator` с методом `createWagon(double capacityTons)` и конкретные создатели для каждого типа.
4. Вызывающий код (`main`) работает **только** с типами `WagonCreator` и `Wagon` и не содержит ни одного `if` по типу вагона.
5. Четвёртый тип (`HopperWagon`, зерновоз) добавляется **новым классом**, без правки ранее написанных классов; это должно быть видно по коду.
6. Проверки в файле (метод `check`, как в образце), минимум четыре:
   * описание каждого из трёх типов не пустое и содержит грузоподъёмность;
   * тип созданного объекта соответствует ожидаемому создателю;
   * четвёртый тип работает наравне с остальными;
   * метод `main` не содержит разбора типов (проверяется глазами преподавателя, в код выносить не нужно).
7. При неуспешной проверке — `throw new AssertionError(...)`.

**Подсказка по структуре файла:** первым идёт `public class FactoryAfter` с `main`, после него — интерфейсы и классы паттерна (как в `TariffAfter.java`).

In [ ]:
%%writefile FactoryAfter.java
// Вариант 9. Пункт таможенного оформления: фабрика создаёт тип декларации.
// Вызывающий код работает с DeclarationCreator и Declaration и не разбирает тип строкой.
public class FactoryAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {
        DeclarationCreator importCreator = new ImportDeclarationCreator();
        DeclarationCreator exportCreator = new ExportDeclarationCreator();
        DeclarationCreator transitCreator = new TransitDeclarationCreator();

        Declaration importDeclaration = importCreator.createDeclaration(1250.0);
        Declaration exportDeclaration = exportCreator.createDeclaration(860.5);
        Declaration transitDeclaration = transitCreator.createDeclaration(430.0);

        System.out.println("--- Декларации, собранные через фабричный метод ---");
        System.out.println(importDeclaration.description());
        System.out.println(exportDeclaration.description());
        System.out.println(transitDeclaration.description());

        System.out.println();
        System.out.println("--- Проверки трёх основных типов ---");
        check("импортная декларация описана и содержит таможенную стоимость",
                describedWithValue(importDeclaration, 1250.0));
        check("экспортная декларация описана и содержит таможенную стоимость",
                describedWithValue(exportDeclaration, 860.5));
        check("транзитная декларация описана и содержит таможенную стоимость",
                describedWithValue(transitDeclaration, 430.0));
        check("создатель импортной декларации вернул ImportDeclaration",
                importDeclaration instanceof ImportDeclaration);
        check("создатель экспортной декларации вернул ExportDeclaration",
                exportDeclaration instanceof ExportDeclaration);
        check("создатель транзитной декларации вернул TransitDeclaration",
                transitDeclaration instanceof TransitDeclaration);

        System.out.println();
        System.out.println("--- Четвёртый тип новым классом, старые классы не менялись ---");
        DeclarationCreator temporaryCreator = new TemporaryAdmissionCreator();
        Declaration temporaryDeclaration = temporaryCreator.createDeclaration(210.0);
        System.out.println(temporaryDeclaration.description());
        check("декларация временного ввоза работает наравне с остальными",
                temporaryDeclaration instanceof TemporaryAdmissionDeclaration
                        && describedWithValue(temporaryDeclaration, 210.0));
        check("описания трёх основных типов различаются",
                !importDeclaration.description().equals(exportDeclaration.description())
                        && !exportDeclaration.description().equals(transitDeclaration.description()));

        finishChecks();
    }

    static boolean describedWithValue(Declaration declaration, double customsValue) {
        String text = declaration.description();
        return text != null
                && !text.isBlank()
                && text.contains(String.valueOf(customsValue))
                && declaration.customsValue() == customsValue;
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    static void finishChecks() {
        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }
}

/** Продукт: общая таможенная декларация. Конкретный тип выбирает создатель. */
interface Declaration {
    String description();
    double customsValue();
}

/** Создатель с фабричным методом. Подкласс решает, какой класс декларации создать. */
abstract class DeclarationCreator {
    abstract Declaration createDeclaration(double customsValue);
}

class ImportDeclarationCreator extends DeclarationCreator {
    Declaration createDeclaration(double customsValue) {
        return new ImportDeclaration(customsValue);
    }
}

class ExportDeclarationCreator extends DeclarationCreator {
    Declaration createDeclaration(double customsValue) {
        return new ExportDeclaration(customsValue);
    }
}

class TransitDeclarationCreator extends DeclarationCreator {
    Declaration createDeclaration(double customsValue) {
        return new TransitDeclaration(customsValue);
    }
}

/** Четвёртый тип: добавлен отдельным создателем, ранее написанные классы не правились. */
class TemporaryAdmissionCreator extends DeclarationCreator {
    Declaration createDeclaration(double customsValue) {
        return new TemporaryAdmissionDeclaration(customsValue);
    }
}

class ImportDeclaration implements Declaration {
    private final double customsValue;

    ImportDeclaration(double customsValue) {
        this.customsValue = customsValue;
    }

    public String description() {
        return "Импортная декларация: таможенная стоимость " + customsValue
                + " тыс. руб., ввоз товаров на таможенную территорию";
    }

    public double customsValue() {
        return customsValue;
    }
}

class ExportDeclaration implements Declaration {
    private final double customsValue;

    ExportDeclaration(double customsValue) {
        this.customsValue = customsValue;
    }

    public String description() {
        return "Экспортная декларация: таможенная стоимость " + customsValue
                + " тыс. руб., вывоз товаров с таможенной территории";
    }

    public double customsValue() {
        return customsValue;
    }
}

class TransitDeclaration implements Declaration {
    private final double customsValue;

    TransitDeclaration(double customsValue) {
        this.customsValue = customsValue;
    }

    public String description() {
        return "Транзитная декларация: таможенная стоимость " + customsValue
                + " тыс. руб., перемещение товаров через таможенную территорию";
    }

    public double customsValue() {
        return customsValue;
    }
}

class TemporaryAdmissionDeclaration implements Declaration {
    private final double customsValue;

    TemporaryAdmissionDeclaration(double customsValue) {
        this.customsValue = customsValue;
    }

    public String description() {
        return "Декларация временного ввоза: таможенная стоимость " + customsValue
                + " тыс. руб., товары с обязательством обратного вывоза";
    }

    public double customsValue() {
        return customsValue;
    }
}


In [ ]:
!java FactoryAfter.java

In [ ]:
%%writefile factory.puml
@startuml
title Factory Method: тип таможенной декларации

interface Declaration {
  +description() : String
  +customsValue() : double
}

abstract class DeclarationCreator {
  +{abstract} createDeclaration(customsValue) : Declaration
}

class ImportDeclaration
class ExportDeclaration
class TransitDeclaration
class TemporaryAdmissionDeclaration

class ImportDeclarationCreator
class ExportDeclarationCreator
class TransitDeclarationCreator
class TemporaryAdmissionCreator

Declaration <|.. ImportDeclaration
Declaration <|.. ExportDeclaration
Declaration <|.. TransitDeclaration
Declaration <|.. TemporaryAdmissionDeclaration

DeclarationCreator <|-- ImportDeclarationCreator
DeclarationCreator <|-- ExportDeclarationCreator
DeclarationCreator <|-- TransitDeclarationCreator
DeclarationCreator <|-- TemporaryAdmissionCreator

ImportDeclarationCreator ..> ImportDeclaration : создаёт
ExportDeclarationCreator ..> ExportDeclaration : создаёт
TransitDeclarationCreator ..> TransitDeclaration : создаёт
TemporaryAdmissionCreator ..> TemporaryAdmissionDeclaration : создаёт
@enduml


In [ ]:
!plantuml -tpng factory.puml
from IPython.display import Image
Image("factory.png")

**Вывод по задаче 1** (чем создание через фабричный метод отличается от создания через `new` в вызывающем коде, и когда фабрика излишня):

В `main` нет `new ImportDeclaration` и нет разбора типа. Там только создатель и общий тип `Declaration`, а конкретный `new` стоит в фабричном методе наследника. Поэтому временный ввоз я добавила классами `TemporaryAdmissionDeclaration` и `TemporaryAdmissionCreator`, не правя импорт, экспорт и транзит. Фабрика лишняя, когда тип один и создаётся в одном месте: обычный `new` короче, а отдельная иерархия создателей ничего не разделяет.


---

## 4. Задача 2: Observer

**Сценарий.** Станция фиксирует прибытие поезда и должна оповестить получателей: табло, журнал, рассылку. В коде «до» станция сама знает про всех получателей.

### 4.1 Код «до» (запустить, не изменяя)

In [ ]:
%%writefile ObserverBefore.java
// Код «до»: источник события жёстко связан со всеми получателями.
public class ObserverBefore {

    public static void main(String[] args) {
        Station station = new Station("Гомель");
        station.trainArrived("3022");
        station.trainArrived("6701");
    }
}

class Station {

    private final String stationName;

    Station(String stationName) {
        this.stationName = stationName;
    }

    void trainArrived(String trainNumber) {
        // Любой новый получатель требует правки этого метода
        System.out.println("[табло]  Поезд " + trainNumber + " прибыл на станцию " + stationName);
        System.out.println("[журнал] " + stationName + ": зафиксировано прибытие поезда " + trainNumber);
    }
}

In [ ]:
!java ObserverBefore.java

### 4.2 Задание

Переписать сценарий на **Observer** в файле `ObserverAfter.java`. Требования:

1. Интерфейс наблюдателя `ArrivalListener` с методом `onTrainArrived(String stationName, String trainNumber)`.
2. Субъект `Station` с методами `addListener`, `removeListener`, `trainArrived`; класс `Station` **не знает**, какие именно наблюдатели существуют.
3. Не менее трёх наблюдателей: табло, журнал, рассылка. Журнал должен **накапливать** записи, чтобы их можно было проверить.
4. Четвёртый наблюдатель (например, передача в АСУ) добавляется без правки класса `Station`.
5. Проверки в файле, минимум четыре:
   * после одного события журнал получил ровно одну запись;
   * после отписки наблюдателя его записи больше не появляются;
   * все подписчики получили одно и то же событие;
   * добавление четвёртого наблюдателя не потребовало изменения `Station`.
6. В выводе указать порядок получения события подписчиками и ответить на вопрос в выводе ниже.

**Подсказка:** список наблюдателей удобно хранить в `java.util.ArrayList`, обход — обычным `for`.

In [ ]:
%%writefile ObserverAfter.java
// Вариант 9. Пункт таможенного оформления: событие «груз оформлен».
// Пункт знает только интерфейс ClearanceListener и не знает конкретные классы получателей.
import java.util.ArrayList;
import java.util.List;

public class ObserverAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {
        CustomsPoint customsPoint = new CustomsPoint("ПТО Брест-Северный");
        BoardDisplay boardDisplay = new BoardDisplay();
        ClearanceJournal clearanceJournal = new ClearanceJournal();
        NotificationMailing notificationMailing = new NotificationMailing();

        customsPoint.addListener(boardDisplay);
        customsPoint.addListener(clearanceJournal);
        customsPoint.addListener(notificationMailing);

        System.out.println("--- Груз ДТ-24091. Порядок: табло, журнал, рассылка ---");
        customsPoint.cargoCleared("ДТ-24091");

        System.out.println();
        System.out.println("--- Проверки подписки ---");
        check("после одного события в журнале ровно одна запись",
                clearanceJournal.recordCount() == 1);
        check("табло, журнал и рассылка получили один и тот же груз",
                "ДТ-24091".equals(boardDisplay.lastCargoNumber())
                        && "ДТ-24091".equals(clearanceJournal.lastCargoNumber())
                        && "ДТ-24091".equals(notificationMailing.lastCargoNumber()));

        System.out.println();
        System.out.println("--- Рассылка отписана, груз ДТ-24092 её уже не касается ---");
        customsPoint.removeListener(notificationMailing);
        customsPoint.cargoCleared("ДТ-24092");
        check("после отписки у рассылки новых записей нет",
                notificationMailing.recordCount() == 1
                        && !notificationMailing.hasCargo("ДТ-24092"));
        check("журнал при этом получил второе событие",
                clearanceJournal.recordCount() == 2
                        && clearanceJournal.hasCargo("ДТ-24092"));

        System.out.println();
        System.out.println("--- Четвёртый наблюдатель (АСУ) без правки CustomsPoint ---");
        AsuTransfer asuTransfer = new AsuTransfer();
        customsPoint.addListener(asuTransfer);
        customsPoint.cargoCleared("ДТ-24093");
        check("АСУ получила событие тем же addListener, класс пункта не менялся",
                asuTransfer.recordCount() == 1
                        && "ДТ-24093".equals(asuTransfer.lastCargoNumber())
                        && clearanceJournal.hasCargo("ДТ-24093"));
        check("отписанная рассылка не получила и третье событие",
                !notificationMailing.hasCargo("ДТ-24093"));

        System.out.println();
        System.out.println("Порядок получения события — порядок подписки в списке.");
        System.out.println("ДТ-24091: табло, журнал, рассылка.");
        System.out.println("ДТ-24092: табло, журнал (рассылка уже отписана).");
        System.out.println("ДТ-24093: табло, журнал, АСУ.");

        finishChecks();
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    static void finishChecks() {
        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }
}

/** Наблюдатель: любой получатель события «груз оформлен». */
interface ClearanceListener {
    void onCargoCleared(String pointName, String cargoNumber);
}

/** Субъект. Не знает, какие именно наблюдатели подписаны. */
class CustomsPoint {

    private final String pointName;
    private final List<ClearanceListener> listeners = new ArrayList<>();

    CustomsPoint(String pointName) {
        this.pointName = pointName;
    }

    void addListener(ClearanceListener listener) {
        listeners.add(listener);
    }

    void removeListener(ClearanceListener listener) {
        listeners.remove(listener);
    }

    void cargoCleared(String cargoNumber) {
        for (ClearanceListener listener : listeners) {
            listener.onCargoCleared(pointName, cargoNumber);
        }
    }
}

class BoardDisplay implements ClearanceListener {

    private final List<String> receivedCargo = new ArrayList<>();

    public void onCargoCleared(String pointName, String cargoNumber) {
        receivedCargo.add(cargoNumber);
        System.out.println("[табло]  Груз " + cargoNumber + " оформлен на пункте " + pointName);
    }

    int recordCount() {
        return receivedCargo.size();
    }

    String lastCargoNumber() {
        return receivedCargo.get(receivedCargo.size() - 1);
    }

    boolean hasCargo(String cargoNumber) {
        return receivedCargo.contains(cargoNumber);
    }
}

class ClearanceJournal implements ClearanceListener {

    private final List<String> records = new ArrayList<>();

    public void onCargoCleared(String pointName, String cargoNumber) {
        records.add(pointName + ": зафиксировано оформление груза " + cargoNumber);
        System.out.println("[журнал] " + pointName + ": зафиксировано оформление груза " + cargoNumber);
    }

    int recordCount() {
        return records.size();
    }

    String lastCargoNumber() {
        String lastRecord = records.get(records.size() - 1);
        int marker = lastRecord.lastIndexOf(' ');
        return lastRecord.substring(marker + 1);
    }

    boolean hasCargo(String cargoNumber) {
        for (String record : records) {
            if (record.contains(cargoNumber)) {
                return true;
            }
        }
        return false;
    }
}

class NotificationMailing implements ClearanceListener {

    private final List<String> receivedCargo = new ArrayList<>();

    public void onCargoCleared(String pointName, String cargoNumber) {
        receivedCargo.add(cargoNumber);
        System.out.println("[рассылка] Груз " + cargoNumber + " оформлен, пункт " + pointName);
    }

    int recordCount() {
        return receivedCargo.size();
    }

    String lastCargoNumber() {
        return receivedCargo.get(receivedCargo.size() - 1);
    }

    boolean hasCargo(String cargoNumber) {
        return receivedCargo.contains(cargoNumber);
    }
}

/** Четвёртый наблюдатель. Подключается через addListener, CustomsPoint не меняется. */
class AsuTransfer implements ClearanceListener {

    private final List<String> receivedCargo = new ArrayList<>();

    public void onCargoCleared(String pointName, String cargoNumber) {
        receivedCargo.add(cargoNumber);
        System.out.println("[АСУ]    В автоматизированную систему передано: "
                + pointName + ", груз " + cargoNumber);
    }

    int recordCount() {
        return receivedCargo.size();
    }

    String lastCargoNumber() {
        return receivedCargo.get(receivedCargo.size() - 1);
    }

    boolean hasCargo(String cargoNumber) {
        return receivedCargo.contains(cargoNumber);
    }
}


In [ ]:
!java ObserverAfter.java

In [ ]:
%%writefile observer.puml
@startuml
title Observer: груз оформлен

interface ClearanceListener {
  +onCargoCleared(pointName, cargoNumber) : void
}

class CustomsPoint {
  +addListener(listener)
  +removeListener(listener)
  +cargoCleared(cargoNumber)
}

class BoardDisplay
class ClearanceJournal
class NotificationMailing
class AsuTransfer

CustomsPoint o--> "0..*" ClearanceListener : оповещает
ClearanceListener <|.. BoardDisplay
ClearanceListener <|.. ClearanceJournal
ClearanceListener <|.. NotificationMailing
ClearanceListener <|.. AsuTransfer
@enduml


In [ ]:
!plantuml -tpng observer.puml
from IPython.display import Image
Image("observer.png")

**Вывод по задаче 2** (какую связанность снял Observer и какие новые проблемы он создал — порядок уведомлений, ошибка в одном подписчике, забытая отписка):

`CustomsPoint` больше не вызывает табло, журнал и рассылку по именам. Он хранит список `ClearanceListener` и при оформлении груза обходит его. АСУ подключается тем же `addListener`, класс пункта я не меняла. Порядок сообщений — это порядок подписки: для ДТ-24091 сначала табло, потом журнал, потом рассылка. Отдельно он нигде не задан, и его легко перепутать, если подписаться в другом порядке. Если один слушатель бросит исключение, цикл прервётся, и следующие получатели событие уже не получат. Если забыть `removeListener`, пункт держит ссылку, и объект не заберёт сборщик мусора. В проверке рассылка отписывается и после этого новых записей не получает.


---

## 5. Сравнение и выводы

Вариант 9, пункт таможенного оформления. Калинина И.А., группа ГИ-41.

### 5.1 Итоговая таблица

| Паттерн | Какую проблему кода «до» устраняет | Что добавляется без правки старого кода | Чем приходится заплатить | Где применим в вашем проекте |
|---|---|---|---|---|
| Strategy | Один метод держит все формулы и ветку `if` на каждый вид. Новый вид без правки этого метода не добавить, чужой вид молча даёт 0 | Новый алгоритм отдельным классом с общим интерфейсом. Калькулятор формул не знает | Больше классов и интерфейс, который надо продумать заранее | Выбор канала контроля: зелёный, жёлтый и красный. Новый канал — новая стратегия, а не ещё один `if` в общем методе |
| Factory Method | Создание объекта и его описание свалены в разбор строки типа. Неизвестный тип превращается в заглушку | Новый тип декларации парой классов: продукт и создатель. `main` по-прежнему видит только `Declaration` и `DeclarationCreator` | Иерархия создателей ради `new`, который раньше был одной строкой | Импортная, экспортная и транзитная декларации. Временный ввоз добавлен новым создателем |
| Observer | Пункт оформления сам печатает табло и журнал и знает каждого получателя | Новый получатель события «груз оформлен» через `addListener`. Класс пункта не меняется | Порядок уведомления нигде не задан отдельно, сбой одного подписчика обрывает остальных, забытая отписка держит объект в памяти | Табло пункта, журнал оформления, рассылка и передача в АСУ |

### 5.2 Контрольные вопросы

1. Какой принцип SOLID нарушает разбор типа условным оператором и как каждый из трёх паттернов его восстанавливает?
   **Ответ:** В первую очередь принцип открытости/закрытости: работающий код приходится менять, чтобы добавить вариант. Часто рядом страдает и единственная ответственность, потому что один метод и выбирает вариант, и выполняет его. Strategy выносит каждый алгоритм в свой класс за общим интерфейсом. Factory Method оставляет выбор конкретного класса наследнику создателя. Observer заменяет прямые вызовы списком подписчиков. Во всех трёх случаях новый вариант — это новый класс, а не новая ветка в старом методе.
2. Чем Factory Method отличается от статического метода создания (`static factory method`) и от Abstract Factory?
   **Ответ:** Factory Method — метод экземпляра, его переопределяет подкласс, и именно подкласс решает, какой продукт создать. Статический метод создания, например `valueOf`, просто прячет `new` внутри одного класса, наследовать тут нечего. Abstract Factory собирает семейство связанных продуктов (несколько фабричных методов сразу), а Factory Method отвечает за один продукт.
3. Strategy и State имеют почти одинаковую структуру классов. В чём разница по назначению?
   **Ответ:** Strategy подменяет алгоритм: клиент сам передаёт нужный способ расчёта, и контекст может вообще не знать, что стратегия сменилась. State описывает состояние объекта. Поведение меняется, потому что состояние стало другим, и переход часто делает сам объект, а не внешний код.
4. Почему Observer может привести к утечке памяти и как этого избежать?
   **Ответ:** Субъект хранит ссылки на подписчиков. Пока пункт оформления жив и держит слушателя в списке, сборщик мусора этот объект не заберёт, даже если остальная программа про него уже забыла. Нужно вызывать `removeListener`, когда получатель больше не нужен. В этой работе так отписывается рассылка: после `removeListener` новые грузы ей не приходят.
5. Приведите пример, когда введение паттерна **вредно**. Что такое преждевременное усложнение?
   **Ответ:** Если в пункте оформления одна декларация и второго типа не предвидится, фабрика из создателя и продукта только прячет обычный `new`. То же с одним получателем события: проще вызвать его напрямую, чем заводить список слушателей. Преждевременное усложнение — это вводить паттерн «на вырост», пока в коде нет реальной развилки и никто не просил её расширять.

### 5.3 Три вывода по работе в целом

1. На образце Strategy рефрижераторная отправка в коде «до» дала 0.00: метод про неё не знал. После выноса формул в стратегии тот же `TariffCalculator` считает новый вид, а `legacyCost` показывает, что старые суммы не изменились.
2. В варианте 9 новый тип декларации добавляется классами фабрики, а новый получатель сообщения «груз оформлен» — подпиской. Ни пункт оформления, ни уже написанные декларации для этого править не пришлось.
3. Паттерн окупается, когда вариантов несколько и они будут добавляться. Пока вариант один, прямой код короче и честнее.


---

## 6. Варианты предметных областей

Номер варианта — по номеру студента в списке группы. Образец (раздел 2) выполняется всеми одинаково; задачи 1 и 2 — на своей предметной области.

| № | Предметная область | Фабрика создаёт | Стратегия варьирует | Наблюдатели получают событие |
|---|---|---|---|---|
| 1 | Грузовая станция | тип вагона | расчёт платы за пользование вагонами | вагон подан под погрузку |
| 2 | Локомотивное депо | вид ремонта (ТО-2, ТР-1, ТР-2) | приоритет заявок на ремонт | локомотив выдан из ремонта |
| 3 | Пассажирская касса | тип проездного документа | начисление скидки | рейс задержан |
| 4 | Контейнерный терминал | тип контейнера | выбор места на площадке | контейнер прибыл |
| 5 | Автотранспортное предприятие | тип транспортного средства | расчёт расхода топлива | путевой лист закрыт |
| 6 | Склад | тип тары | правило размещения (FIFO, ABC) | запас ниже нормы |
| 7 | Сортировочная станция | тип отцепа | очерёдность расформирования | состав расформирован |
| 8 | Пригородные перевозки | тип абонемента | тарификация по зонам | изменено расписание |
| 9 | Пункт таможенного оформления | тип декларации | выбор канала контроля | груз оформлен |
| 10 | Экспедиторская компания | тип заявки (авто, ж. д., море) | выбор перевозчика по критерию | статус отправки изменён |
| 11 | Путевое хозяйство | тип дефекта рельса | назначение вида работ | обнаружен дефект |
| 12 | Энергодиспетчерский пункт | тип потребителя | тарифный план (день, ночь) | авария на подстанции |
| 13 | Вагонное депо | тип неисправности | выбор отцепочного или безотцепочного ремонта | вагон забракован |
| 14 | Логистический центр | тип упаковки | правило консолидации отправок | сформирована отправка |
| 15 | Центр фирменного транспортного обслуживания | тип договора | расчёт скидки за объём | заявка согласована |

---

## 7. Чек-лист самопроверки

- [x] Раздел 1 выполнен, версия Java не ниже 11
- [x] `TariffBefore.java` и `TariffAfter.java` запущены, вывод сохранён
- [x] Отвечены вопросы к образцу (п. 2.5)
- [x] `FactoryAfter.java` написан, все проверки `[OK]`, в `main` нет разбора типов
- [x] Четвёртый тип декларации добавлен новым классом
- [x] `ObserverAfter.java` написан, все проверки `[OK]`, отписка работает
- [x] Четвёртый наблюдатель добавлен без правки класса пункта оформления
- [x] Построены три диаграммы: `strategy.puml`, `factory.puml`, `observer.puml`
- [x] Заполнена итоговая таблица (п. 5.1)
- [x] Отвечены пять контрольных вопросов и сформулированы три вывода
- [ ] Блокнот выполняется сверху вниз без ошибок после *Среда выполнения → Перезапустить и выполнить все*
- [ ] Запущена ячейка самопроверки, её вывод вставлен в описание pull request


In [1]:
#@title Запустить самопроверку и получить отчёт
import os, re, subprocess, datetime

ФИО_исполнителя = "Калинина И.А."  #@param {type:"string"}
Группа_исполнителя = "ГИ-41"  #@param {type:"string"}
Номер_варианта = 9  #@param {type:"integer"}
ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ПРОГРАММЫ = [
    ("Образец, Strategy", "TariffAfter.java"),
    ("Задача 1, Factory Method", "FactoryAfter.java"),
    ("Задача 2, Observer", "ObserverAfter.java"),
]
ДИАГРАММЫ = ["strategy.puml", "factory.puml", "observer.puml"]

def запустить(имя_файла):
    """Запустить один исходный файл Java и вернуть (успех, пояснение)."""
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    if завершение.returncode != 0:
        значимые = [s for s in вывод.splitlines() if s.strip() and not s.startswith("Picked up")]
        первая_строка = значимые[0] if значимые else "без сообщения"
        return False, "аварийное завершение: " + первая_строка[:90]
    if "[FAIL]" in вывод:
        return False, "проверок не пройдено: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 3:
        return False, "проверок в файле только %d, требуется не менее 3" % успешных
    return True, "проверок пройдено: %d" % успешных

результаты = []
for название, файл in ПРОГРАММЫ:
    результаты.append((название, файл) + запустить(файл))

# Признаки соблюдения требований, которые можно проверить по тексту
замечания = []
def тело_метода(исходный_текст, сигнатура="main("):
    """Вернуть тело метода по сопоставлению фигурных скобок (без построчных комментариев)."""
    без_комментариев = re.sub(r"//[^\n]*", "", исходный_текст)
    начало = без_комментариев.find(сигнатура)
    if начало < 0:
        return ""
    открывающая = без_комментариев.find("{", начало)
    if открывающая < 0:
        return ""
    глубина = 0
    for позиция in range(открывающая, len(без_комментариев)):
        символ = без_комментариев[позиция]
        if символ == "{":
            глубина += 1
        elif символ == "}":
            глубина -= 1
            if глубина == 0:
                return без_комментариев[открывающая:позиция + 1]
    return без_комментариев[открывающая:]

if os.path.exists("FactoryAfter.java"):
    текст_фабрики = open("FactoryAfter.java", encoding="utf-8").read()
    блок_main = тело_метода(текст_фабрики)
    if re.search(r'\b(if|switch)\b', блок_main):
        замечания.append("в FactoryAfter.java метод main содержит ветвление по типу")
    if текст_фабрики.count("class ") < 5:
        замечания.append("в FactoryAfter.java меньше пяти классов: продукты, создатели и четвёртый тип, вероятно, не описаны")
if os.path.exists("ObserverAfter.java"):
    текст_наблюдателя = open("ObserverAfter.java", encoding="utf-8").read()
    for требуемое in ("addListener", "removeListener"):
        if требуемое not in текст_наблюдателя:
            замечания.append("в ObserverAfter.java нет метода %s" % требуемое)
отсутствующие_диаграммы = [d for d in ДИАГРАММЫ if not os.path.exists(d)]

строки = [
    "### Отчёт самопроверки «Песочница паттернов GoF»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s, вариант %d |" % (
        ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА", Номер_варианта),
    "| Дата и время отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "| Версия Java | %s |" % next(
        (s for s in subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()
         if "version" in s), "не определена"),
    "",
    "| Программа | Файл | Результат |",
    "|---|---|---|",
]
for название, файл, успех, пояснение in результаты:
    строки.append("| %s | `%s` | %s — %s |" % (название, файл, "пройдено" if успех else "НЕ ПРОЙДЕНО", пояснение))
строки += [
    "",
    "| Диаграммы | %s |" % ("все три на месте" if not отсутствующие_диаграммы
                            else "отсутствуют: " + ", ".join(отсутствующие_диаграммы)),
    "| Таблица и выводы | %s |" % ("заполнены" if ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
    "| Замечания | %s |" % ("нет" if not замечания else "; ".join(замечания)),
]
print("\n".join(строки))
print("\nСкопируйте отчёт выше в описание pull request.\n")

ошибки = []
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
ошибки += ["%s: %s" % (название, пояснение) for название, файл, успех, пояснение in результаты if not успех]
if отсутствующие_диаграммы:
    ошибки.append("нет диаграмм: " + ", ".join(отсутствующие_диаграммы))
if not ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены итоговая таблица и выводы")
ошибки += замечания

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")


### Отчёт самопроверки «Песочница паттернов GoF»

| Параметр | Значение |
|---|---|
| Исполнитель | Калинина И.А., группа ГИ-41, вариант 9 |
| Дата и время отчёта (UTC) | 2026-11-14 20:14 |
| Версия Java | openjdk version "11.0.16.1" 2022-08-12 LTS |

| Программа | Файл | Результат |
|---|---|---|
| Образец, Strategy | `TariffAfter.java` | пройдено — проверок пройдено: 5 |
| Задача 1, Factory Method | `FactoryAfter.java` | пройдено — проверок пройдено: 8 |
| Задача 2, Observer | `ObserverAfter.java` | пройдено — проверок пройдено: 6 |

| Диаграммы | все три на месте |
| Таблица и выводы | заполнены |
| Замечания | нет |

Скопируйте отчёт выше в описание pull request.

Самопроверка пройдена: работа готова к сдаче.


---

## 9. Критерии оценивания

| Критерий | Баллов |
|---|---|
| Образец запущен, вопросы п. 2.5 отвечены содержательно | 10 |
| Factory Method: корректная структура паттерна, в `main` нет разбора типов | 20 |
| Factory Method: четвёртый тип добавлен новым классом | 10 |
| Observer: корректная структура, работают подписка и отписка | 20 |
| Observer: четвёртый наблюдатель добавлен без правки субъекта | 10 |
| Проверки в каждом файле осмысленны (не `[OK]` ради `[OK]`) | 10 |
| Три диаграммы классов соответствуют коду | 10 |
| Итоговая таблица, контрольные вопросы, выводы | 10 |
| **Итого** | **100** |

Снижение: задача решена не на своём варианте — минус 20; код «после» повторяет разбор типа условным оператором внутри нового класса — минус 20; диаграмма не соответствует коду — минус 10.

## 10. Для желающих (сверх задания)

| Что сделать | Зачем |
|---|---|
| Подключить настоящий JUnit 5 вместо метода `check`: `!wget -q https://repo1.maven.org/maven2/org/junit/platform/junit-platform-console-standalone/1.10.2/junit-platform-console-standalone-1.10.2.jar` и запуск через `java -jar` | переход к инструменту, который используется в ЛР по модульному тестированию |
| Заменить три `if` в выборе стратегии на `java.util.Map` с ключом-перечислением (`enum`) | показать, что каталог стратегий сам может быть устранён из вызывающего кода |
| Реализовать Observer на `java.util.function.Consumer` вместо своего интерфейса | сравнить классический GoF и функциональный стиль |